In [0]:
import pandas as pd

In [0]:
spark.sql("use catalog proyecto_final_prueba")

DataFrame[]

In [0]:
catalog = spark.sql("select current_catalog()").first()[0]
schema = "gold"
table = "dim_ubicacion"

In [0]:
spark.sql(f"create schema if not exists {catalog}.{schema}")

DataFrame[]

In [0]:
spark.sql(f"drop table if exists {catalog}.{schema}.{table}")

DataFrame[]

DataFrame[]

In [0]:
df_silver = spark.table(f"{catalog}.silver.weather").toPandas()
df_silver


,fecha_hora_local,latitude,longitude,temperature,humidity,wind_speed,weather_code,weather_desc
0,2026-06-01 03:00:00,-12.056238,-77.06198,19.0,93,8.0,3,Nublado
1,2026-06-01 04:00:00,-12.056238,-77.06198,19.0,92,8.1,3,Nublado
2,2026-06-01 05:00:00,-12.056238,-77.06198,18.8,92,8.3,3,Nublado
3,2026-06-01 06:00:00,-12.056238,-77.06198,19.1,89,9.1,3,Nublado
4,2026-06-01 07:00:00,-12.056238,-77.06198,19.4,90,5.5,3,Nublado
...,...,...,...,...,...,...,...,...
5443,2026-03-24 06:00:00,-12.056238,-77.06198,21.3,83,1.4,3,Nublado
5444,2026-03-24 07:00:00,-12.056238,-77.06198,21.7,80,5.5,3,Nublado
5445,2026-03-24 08:00:00,-12.056238,-77.06198,22.3,80,6.0,3,Nublado
5446,2026-03-24 09:00:00,-12.056238,-77.06198,22.9,81,6.6,1,Nublado


In [0]:
dim = pd.DataFrame(df_silver[["latitude", "longitude"]]).drop_duplicates().reset_index(drop=True)
dim["id_ubicacion"] = dim.index+1
dim["nombre_lugar"] = 'Lima'
columns = ["id_ubicacion", "latitude", "longitude", "nombre_lugar"]
dim = dim[columns]
dim

,id_ubicacion,latitude,longitude,nombre_lugar
0,1,-12.056238,-77.06198,Lima


In [0]:
df_spark = spark.createDataFrame(dim)
df_spark.display()

id_ubicacion,latitude,longitude,nombre_lugar
1,-12.056238,-77.06198,Lima


In [0]:
df_spark.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.{table}")